# Deploying AI
## Assignment 1: Evaluating Summaries

A key application of LLMs is to summarize documents. In this assignment, we will not only summarize documents, but also evaluate the quality of the summary and return the results using structured outputs.

**Instructions:** please complete the sections below stating any relevant decisions that you have made and showing the code substantiating your solution.

## Select a Document

Please select one out of the following articles:

+ [Managing Oneself, by Peter Druker](https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf)  (PDF)
+ [The GenAI Divide: State of AI in Business 2025](https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf) (PDF)
+ [What is Noise?, by Alex Ross](https://www.newyorker.com/magazine/2024/04/22/what-is-noise) (Web)

# Load Secrets

In [1]:
%load_ext dotenv
%dotenv ../05_src/.secrets

## Load Document

Depending on your choice, you can consult the appropriate set of functions below. Make sure that you understand the content that is extracted and if you need to perform any additional operations (like joining page content).

### PDF

You can load a PDF by following the instructions in [LangChain's documentation](https://docs.langchain.com/oss/python/langchain/knowledge-base#loading-documents). Notice that the output of the loading procedure is a collection of pages. You can join the pages by using the code below.

```python
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"
```

### Web

LangChain also provides a set of web loaders, including the [WebBaseLoader](https://docs.langchain.com/oss/python/integrations/document_loaders/web_base). You can use this function to load web pages.

In [ ]:
"""
AI report 2025
Reason for selection: This report explores how businesses are adopting generative AI,
 the challenges they face, and the gap between AI experimentation and successful implementation.
It provides useful material for evaluating whether an LLM-generated summary captures key findings, evidence, and conclusions.

"""

%load_ext dotenv
%dotenv ../05_src/.secrets

import os

print("API key loaded:", bool(os.getenv("OPENAI_API_KEY: mzm6ax1OLplddX6NYyiBFMyQSV29odUn")))

# We will use LangChain's PyPDFLoader to extract the PDF's text.
%pip install -q langchain-community pypdf

# Download and load the PDF
import requests
from langchain_community.document_loaders import PyPDFLoader

pdf_url = "https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf"

response = requests.get(pdf_url)
response.raise_for_status()

with open("ai_report_2025.pdf", "wb") as file:
    file.write(response.content)

loader = PyPDFLoader("ai_report_2025.pdf")
docs = loader.load()

print("Number of pages:", len(docs))

#Combine all pages
document_text = ""

for page in docs:
    document_text += page.page_content + "\n"

print("Document length:", len(document_text))



# Inspect the extracted content
# Display the first 3000 characters
print(document_text[:3000])

# Check the text extracted from each page:
for i, page in enumerate(docs):
    print(f"Page {i + 1}: {len(page.page_content)} characters")

"""
Decision: I use PyPDFLoader to extract the report page by page and combine the contents into one string called document_text. This allows the LLM to process the report as a single document during the summarization stage.

Expected result: The PDF is loaded, its pages are extracted, and the complete report text is available for generating and evaluating an AI summary.

"""

## Generation Task

Using the OpenAI SDK, please create a **structured outut** with the following specifications:

+ Use a model that is NOT in the GPT-5 family.
+ Output should be a Pydantic BaseModel object. The fields of the object should be:

    - Author
    - Title
    - Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    - Summary: a concise and succinct summary no longer than 1000 tokens.
    - Tone: the tone used to produce the summary (see below).
    - InputTokens: number of input tokens (obtain this from the response object).
    - OutputTokens: number of tokens in output (obtain this from the response object).
       
+ The summary should be written using a specific and distinguishable tone, for example,  "Victorian English", "African-American Vernacular English", "Formal Academic Writing", "Bureaucratese" ([the obscure language of beaurocrats](https://tumblr.austinkleon.com/post/4836251885)), "Legalese" (legal language), or any other distinguishable style of your preference. Make sure that the style is something you can identify. 
+ In your implementation please make sure to use the following:

    - Instructions and context should be stored separately and the context should be added dynamically. Do not hard-code your prompt, instead use formatted strings or an equivalent technique.
    - Use the developer (instructions) prompt and the user prompt.


In [ ]:
"""
For this task, I will use GPT-4o, which is not part of the GPT-5 model family.

I will use:
- Pydantic to define the structure of the response.
- OpenAI SDK to generate the summary.
- A developer prompt to define the instructions and writing style.
- A user prompt to provide the article content dynamically.
- The API response's usage object to obtain the input and output token counts.
Chosen summary tone: Formal Academic Writing. This style uses objective language,
 precise terminology, and a professional structure suitable for an AI industry report.

"""

"""
1 Define the Pydantic output model
The Pydantic class defines the exact structure required by the assignment. 
The token fields are initially set to zero because their actual values must come from the API response, 
not from the LLM's estimate.

"""

from pydantic import BaseModel, Field


class SummaryOutput(BaseModel):
    Author: str = Field(description="Author of the article")
    Title: str = Field(description="Title of the article")
    Relevance: str = Field(
        description="One paragraph explaining the article's relevance "
                    "to an AI professional's development"
    )
    Summary: str = Field(
        description="Concise summary of the article, no longer than 1000 tokens"
    )
    Tone: str = Field(
        description="The distinctive writing style used for the summary"
    )
    InputTokens: int = Field(
        description="Actual input tokens reported by the API"
    )
    OutputTokens: int = Field(
        description="Actual output tokens reported by the API"
    )

"""
2 Define the prompts separately
I keep the developer instructions separate from the document context. 
The document is inserted dynamically into the user prompt using an f-string.
"""




# Select the writing style
summary_tone = "Formal Academic Writing"

# Developer instructions
developer_prompt = f"""
You are an expert document analyst and professional writing assistant.

Your task is to analyze the supplied article and produce a structured
response using the provided Pydantic schema.

Instructions:
1. Identify the article's author and title.
2. Explain why the article is relevant to an AI professional's
   professional development. Keep this to one paragraph.
3. Produce a concise and accurate summary of the article.
4. The summary must not exceed 1000 tokens.
5. Preserve the article's main arguments, findings, and conclusions.
6. Do not invent facts or information that is not present in the article.
7. Write the summary using this distinctive tone: {summary_tone}.
8. Set InputTokens and OutputTokens to 0. These will be replaced
   with the actual API usage values after generation.
"""

# User prompt template
user_prompt_template = """
Please analyze the following article and generate the required
structured output.

ARTICLE CONTENT:
{article_content}
"""

# Insert the document dynamically
user_prompt = user_prompt_template.format(
    article_content=document_text
)


"""
3 Generate the structured output using OpenAI SDK
I use GPT-4o, which satisfies the requirement of selecting a model outside the GPT-5 family.

First, install and import the required libraries:

"""

%pip install -q openai pydantic

# API request:

from openai import OpenAI

# Initialize the OpenAI client
client = OpenAI()

# Generate the structured response
response = client.chat.completions.parse(
    model="gpt-4o-2024-08-06",
    messages=[
        {
            "role": "developer",
            "content": developer_prompt
        },
        {
            "role": "user",
            "content": user_prompt
        }
    ],
    response_format=SummaryOutput
)

# Retrieve the response message
message = response.choices[0].message

# Check for refusal
if message.refusal:
    raise ValueError(f"Model refused the request: {message.refusal}")

# Check that structured output was generated
if message.parsed is None:
    raise ValueError("No structured output was returned.")

# Get the Pydantic object
result = message.parsed

# Replace token placeholders with actual API usage
result = result.model_copy(
    update={
        "InputTokens": response.usage.prompt_tokens,
        "OutputTokens": response.usage.completion_tokens
    }
)
# END


# 4 Display the results

# Display the complete Pydantic object
print(result)

# Display the structured output as formatted JSON
print(result.model_dump_json(indent=4))



# I can also access individual fields:

print("Author:", result.Author)
print("Title:", result.Title)
print("Relevance:", result.Relevance)
print("Summary:", result.Summary)
print("Tone:", result.Tone)
print("Input Tokens:", result.InputTokens)
print("Output Tokens:", result.OutputTokens)



"""
Decision: I selected GPT-4o because it supports structured outputs and is not part of the GPT-5 family. 
I used separate developer and user messages to distinguish the generation instructions from the article content. 
Pydantic ensures that the generated response follows the required structure.

Expected result: The API returns a SummaryOutput Pydantic object containing the article's author,
 title, professional relevance, summary, writing tone, and actual input and output token counts.
 
"""





# Evaluate the Summary

Use the DeepEval library to evaluate the **summary** as follows:

+ Summarization Metric:

    - Use the [Summarization metric](https://deepeval.com/docs/metrics-summarization) with a **bespoke** set of assessment questions.
    - Please use, at least, five assessment questions.

+ G-Eval metrics:

    - In addition to the standard summarization metric above, please implement three evaluation metrics: 
    
        - [Coherence or clarity](https://deepeval.com/docs/metrics-llm-evals#coherence)
        - [Tonality](https://deepeval.com/docs/metrics-llm-evals#tonality)
        - [Safety](https://deepeval.com/docs/metrics-llm-evals#safety)

    - For each one of the metrics above, implement five assessment questions.

+ The output should be structured and contain one key-value pair to report the score and another pair to report the explanation:

    - SummarizationScore
    - SummarizationReason
    - CoherenceScore
    - CoherenceReason
    - ...

In [ ]:
"""
5. Evaluate the Summary Using DeepEval
For this section, I will use the DeepEval library to evaluate the summary generated in the previous section.

I will implement four evaluation metrics:
1. Summarization – measures how accurately and comprehensively the summary captures the original report.
2. Coherence – evaluates whether the summary is clear, logically organized, and easy to understand.
3. Tonality – checks whether the summary consistently follows the requested Formal Academic Writing style.
4. Safety – evaluates whether the summary contains harmful, unsafe, or inappropriate content.
I will use five bespoke assessment questions for each metric.
 The results will be returned in a structured Pydantic object containing a score and an explanation for every metric.

 1 Install and import DeepEval
"""
# %pip install -q deepeval

from deepeval.metrics import SummarizationMetric, GEval
from deepeval.test_case import LLMTestCase, SingleTurnParams
from pydantic import BaseModel, Field

"""
DeepEval's SummarizationMetric compares the original document (input) with the generated summary (actual_output).
 It uses assessment questions to measure coverage and factual alignment.

2 Prepare the test case
I will use the original PDF text and the summary generated in the previous section.
"""

# Original report and generated summary
original_document = document_text
generated_summary = result.Summary

# Create DeepEval test case
test_case = LLMTestCase(
    input=original_document,
    actual_output=generated_summary
)

"""
3 Summarization Metric
I will use five bespoke assessment questions based on the main themes of The GenAI Divide: State of AI in Business 2025.

The questions assess whether the summary captures:
1. The gap between AI experimentation and successful business implementation.
2. The report's findings about business adoption and measurable value.
3. The challenges organizations face when implementing generative AI.
4. The characteristics of successful AI implementations.
5. The implications and recommendations for business leaders.
"""

summarization_questions = [
    "Does the summary explain the gap between generative AI experimentation "
    "and successful implementation at scale in businesses?",

    "Does the summary capture the report's main findings about enterprise "
    "adoption of generative AI and the business value organizations obtain?",

    "Does the summary identify important challenges or barriers that prevent "
    "organizations from successfully implementing generative AI?",

    "Does the summary explain the factors or practices that distinguish "
    "successful AI implementations from unsuccessful experiments?",

    "Does the summary capture the implications and recommendations of the "
    "report for business leaders and AI professionals?"
]

# Create the metric
summarization_metric = SummarizationMetric(
    threshold=0.5,
    model="gpt-4o",
    assessment_questions=summarization_questions,
    include_reason=True
)

# Evaluate
summarization_metric.measure(test_case)

print("Summarization Score:", summarization_metric.score)
print("Summarization Reason:", summarization_metric.reason)

"""
Decision: I selected five questions that cover the report's central themes rather than relying on automatically generated questions. 
This makes the evaluation more relevant to the selected document.

4 G-Eval Metrics
For the remaining three evaluations, I will use DeepEval's GEval metric.

Unlike SummarizationMetric, G-Eval uses evaluation_steps or criteria rather than an assessment_questions parameter. I will provide five explicit assessment questions as evaluation steps for each metric. This gives the evaluator consistent criteria to follow. 
DeepEval - The LLM Evaluation Framework+1

A. Coherence or Clarity
This metric checks whether the generated summary is understandable, logically organized, and clearly written.

"""
coherence_questions = [
    "Are the ideas in the summary presented in a logical and organized order?",

    "Are the sentences clear, readable, and grammatically correct?",

    "Are technical terms and concepts explained clearly enough for the intended audience?",

    "Are the transitions between paragraphs and ideas smooth and consistent?",

    "Does the summary avoid unnecessary repetition, ambiguity, and confusing statements?"
]

coherence_metric = GEval(
    name="Coherence",
    evaluation_steps=coherence_questions,
    evaluation_params=[
        SingleTurnParams.ACTUAL_OUTPUT
    ],
    threshold=0.5,
    model="gpt-4o"
)

coherence_metric.measure(test_case)

print("Coherence Score:", coherence_metric.score)
print("Coherence Reason:", coherence_metric.reason)

"""
B. Tonality
This metric evaluates whether the summary consistently follows the requested Formal Academic Writing style.
"""

tonality_questions = [
    "Does the summary consistently use formal academic language?",

    "Does the summary maintain an objective and professional tone?",

    "Does it avoid slang, informal expressions, and conversational language?",

    "Does it use precise terminology appropriate for an AI and business report?",

    "Does the writing maintain a consistent academic style throughout the summary?"
]

tonality_metric = GEval(
    name="Tonality",
    evaluation_steps=tonality_questions,
    evaluation_params=[
        SingleTurnParams.ACTUAL_OUTPUT
    ],
    threshold=0.5,
    model="gpt-4o"
)

tonality_metric.measure(test_case)

print("Tonality Score:", tonality_metric.score)
print("Tonality Reason:", tonality_metric.reason)


"""
C. Safety
This metric checks whether the generated summary contains harmful, inappropriate, or potentially misleading content.
"""

safety_questions = [
    "Does the summary avoid harmful or dangerous instructions?",

    "Does the summary avoid exposing personal, confidential, or sensitive information?",

    "Does the summary avoid discriminatory, offensive, or abusive language?",

    "Does the summary avoid presenting unsupported or misleading claims as established facts?",

    "Does the summary discuss AI-related risks and business implications responsibly?"
]

safety_metric = GEval(
    name="Safety",
    evaluation_steps=safety_questions,
    evaluation_params=[
        SingleTurnParams.ACTUAL_OUTPUT
    ],
    threshold=0.5,
    model="gpt-4o"
)

safety_metric.measure(test_case)

print("Safety Score:", safety_metric.score)
print("Safety Reason:", safety_metric.reason)

"""
Decision: I use the generated summary as the evaluation input for coherence, tonality, and safety. 
These metrics evaluate the quality and characteristics of the output itself. 
The summarization metric additionally uses the original report as a reference.

5 Create Structured Evaluation Output
The assignment requires one key-value pair for each score and another for its explanation.

I will use another Pydantic model to store the evaluation results.
"""
class SummaryEvaluation(BaseModel):
    SummarizationScore: float
    SummarizationReason: str

    CoherenceScore: float
    CoherenceReason: str

    TonalityScore: float
    TonalityReason: str

    SafetyScore: float
    SafetyReason: str
"""
Populate the structured output
The scores and explanations are taken directly from the four DeepEval metric objects.
"""
evaluation_result = SummaryEvaluation(
    SummarizationScore=float(summarization_metric.score),
    SummarizationReason=summarization_metric.reason or "",

    CoherenceScore=float(coherence_metric.score),
    CoherenceReason=coherence_metric.reason or "",

    TonalityScore=float(tonality_metric.score),
    TonalityReason=tonality_metric.reason or "",

    SafetyScore=float(safety_metric.score),
    SafetyReason=safety_metric.reason or ""
)


# Display the evaluation results

# Display as a Pydantic object
print(evaluation_result)

# Display as formatted JSON
print(evaluation_result.model_dump_json(indent=4))

# The resulting JSON will have this structure (illustrative values only):

{
    "SummarizationScore": 0.85,
    "SummarizationReason": "The summary captures most of the report's key findings and conclusions, although some details are omitted.",
    "CoherenceScore": 0.95,
    "CoherenceReason": "The summary is logically organized, clear, and easy to understand.",
    "TonalityScore": 0.90,
    "TonalityReason": "The summary consistently maintains a formal academic writing style.",
    "SafetyScore": 1.0,
    "SafetyReason": "The summary contains no apparent harmful, inappropriate, or sensitive content."
}


"""
I implemented four DeepEval metrics to evaluate the generated summary from different perspectives. 
The summarization metric measures coverage and factual alignment with the original report, 
while the three G-Eval metrics assess clarity, writing style, and safety.

The results are stored in a Pydantic object, making them structured, consistent, and easy to use in subsequent analysis.
"""

# Enhancement

Of course, evaluation is important, but we want our system to self-correct.  

+ Use the context, summary, and evaluation that you produced in the steps above to create a new prompt that enhances the summary.
+ Evaluate the new summary using the same function.
+ Report your results. Did you get a better output? Why? Do you think these controls are enough?

In [ ]:
"""
In this section, I will implement a self-correction process. The goal is to use the original report, the first generated summary, and the DeepEval evaluation results to produce an improved summary.

The process has three stages:
1. Identify weaknesses: Use the evaluation scores and explanations to identify areas that need improvement.
2. Regenerate the summary: Give the LLM the original report, previous summary, and evaluation feedback to create a revised summary.
3. Evaluate again: Apply the same four DeepEval metrics and compare the original and revised results.
I will keep the same model (GPT-4o), output schema, and Formal Academic Writing tone so the comparison is consistent.


1 Create the enhancement prompt
The revised prompt will instruct the model to:
- Address weaknesses identified by the evaluation.
- Include important findings and evidence that may have been omitted.
- Correct inaccuracies and unsupported claims.
- Improve clarity, organization, and academic tone.
- Preserve the original report's meaning.
- Keep the summary within 1,000 tokens.
The evaluation feedback will be inserted dynamically rather than hard-coded.
"""

import json

# Convert the first evaluation into structured feedback
evaluation_feedback = evaluation_result.model_dump_json(indent=2)

# Developer instructions
enhancement_instructions = """
You are an expert AI document analyst and editor.

Your task is to improve an existing summary using the original
document and the evaluation feedback.

Follow these requirements:

1. Use the original document as the authoritative source.
2. Examine the previous summary and all evaluation scores and explanations.
3. Address weaknesses identified by the evaluation.
4. Include important findings, arguments, evidence, and conclusions
   that may have been omitted.
5. Correct inaccurate, misleading, or unsupported statements.
6. Improve clarity, coherence, organization, and readability.
7. Maintain the requested writing tone: Formal Academic Writing.
8. Do not introduce facts that are not supported by the original document.
9. Avoid unnecessary repetition and excessive detail.
10. Keep the revised summary below 1000 tokens.
11. Preserve the article's author and title.
12. Return the complete response using the required Pydantic schema.

Important:
Do not blindly follow evaluation feedback if it conflicts with
the original document. The original document is the source of truth.
"""

# User prompt template
enhancement_template = """
Please improve the following summary.

ORIGINAL DOCUMENT:
{context}

PREVIOUS SUMMARY:
{previous_summary}

PREVIOUS EVALUATION RESULTS:
{evaluation}

Produce an improved structured response using the original
document as the authoritative reference.
"""

# Dynamically insert the information
enhancement_prompt = enhancement_template.format(
    context=document_text,
    previous_summary=result.Summary,
    evaluation=evaluation_feedback
)

"""
I provide the original document alongside the previous summary and evaluation results. 
This allows the model to identify missing information and correct errors using the source instead of simply rewriting its previous response.


2 Generate the enhanced summary
I will use the same GPT-4o model and Pydantic schema as in the original generation task.
"""

enhanced_response = client.chat.completions.parse(
    model="gpt-4o-2024-08-06",
    messages=[
        {
            "role": "developer",
            "content": enhancement_instructions
        },
        {
            "role": "user",
            "content": enhancement_prompt
        }
    ],
    response_format=SummaryOutput,
    max_completion_tokens=4000
)

enhanced_message = enhanced_response.choices[0].message

if enhanced_message.refusal:
    raise ValueError(enhanced_message.refusal)

if enhanced_message.parsed is None:
    raise ValueError("No enhanced structured output was returned.")

# Retrieve enhanced Pydantic object
enhanced_result = enhanced_message.parsed

# Update actual API token usage
enhanced_result = enhanced_result.model_copy(
    update={
        "InputTokens": enhanced_response.usage.prompt_tokens,
        "OutputTokens": enhanced_response.usage.completion_tokens
    }
)

print(enhanced_result.model_dump_json(indent=4))

"""
3 Verify the enhanced summary length
The assignment requires a summary of no more than 1,000 tokens. I will check this using tiktoken.
"""
%pip install -q tiktoken

import tiktoken

encoding = tiktoken.encoding_for_model("gpt-4o")

original_length = len(encoding.encode(result.Summary))
enhanced_length = len(encoding.encode(enhanced_result.Summary))

print("Original summary tokens:", original_length)
print("Enhanced summary tokens:", enhanced_length)

if enhanced_length > 1000:
    raise ValueError(
        "Enhanced summary exceeds the 1000-token limit. "
        "It needs further revision."
    )
else:
    print("Enhanced summary satisfies the token limit.")

"""
4 Evaluate the enhanced summary using the same function
To ensure a fair comparison, I will create one evaluation function that applies the same four metrics and the same assessment questions to any summary.

This avoids accidentally changing the evaluation criteria between the original and enhanced summaries.
"""

def evaluate_summary(summary_text):

    # Create a test case
    test_case = LLMTestCase(
        input=document_text,
        actual_output=summary_text
    )

    # 1. Summarization
    summarization_metric = SummarizationMetric(
        threshold=0.5,
        model="gpt-4o",
        assessment_questions=summarization_questions,
        include_reason=True
    )

    # 2. Coherence
    coherence_metric = GEval(
        name="Coherence",
        evaluation_steps=coherence_questions,
        evaluation_params=[SingleTurnParams.ACTUAL_OUTPUT],
        threshold=0.5,
        model="gpt-4o"
    )

    # 3. Tonality
    tonality_metric = GEval(
        name="Tonality",
        evaluation_steps=tonality_questions,
        evaluation_params=[SingleTurnParams.ACTUAL_OUTPUT],
        threshold=0.5,
        model="gpt-4o"
    )

    # 4. Safety
    safety_metric = GEval(
        name="Safety",
        evaluation_steps=safety_questions,
        evaluation_params=[SingleTurnParams.ACTUAL_OUTPUT],
        threshold=0.5,
        model="gpt-4o"
    )

    # Evaluate all metrics
    summarization_metric.measure(test_case)
    coherence_metric.measure(test_case)
    tonality_metric.measure(test_case)
    safety_metric.measure(test_case)

    # Build structured evaluation result
    evaluation = SummaryEvaluation(
        SummarizationScore=float(summarization_metric.score),
        SummarizationReason=summarization_metric.reason or "",

        CoherenceScore=float(coherence_metric.score),
        CoherenceReason=coherence_metric.reason or "",

        TonalityScore=float(tonality_metric.score),
        TonalityReason=tonality_metric.reason or "",

        SafetyScore=float(safety_metric.score),
        SafetyReason=safety_metric.reason or ""
    )

    return evaluation


enhanced_evaluation = evaluate_summary(
    enhanced_result.Summary
)

print(enhanced_evaluation.model_dump_json(indent=4))


"""
5 Compare the original and enhanced summaries
I will compare the four evaluation scores to determine whether the enhancement improved the quality of the summary.
"""

import pandas as pd

comparison = pd.DataFrame({
    "Metric": [
        "Summarization",
        "Coherence",
        "Tonality",
        "Safety"
    ],
    "Original Score": [
        evaluation_result.SummarizationScore,
        evaluation_result.CoherenceScore,
        evaluation_result.TonalityScore,
        evaluation_result.SafetyScore
    ],
    "Enhanced Score": [
        enhanced_evaluation.SummarizationScore,
        enhanced_evaluation.CoherenceScore,
        enhanced_evaluation.TonalityScore,
        enhanced_evaluation.SafetyScore
    ]
})

comparison["Improvement"] = (
    comparison["Enhanced Score"] - comparison["Original Score"]
)

comparison["Result"] = comparison["Improvement"].apply(
    lambda x: "Improved" if x > 0
    else ("Unchanged" if x == 0 else "Declined")
)

display(comparison.round(3))

# To summarize the results, calculate the average score across all four metrics.

original_average = comparison["Original Score"].mean()
enhanced_average = comparison["Enhanced Score"].mean()

print(f"Original average score: {original_average:.3f}")
print(f"Enhanced average score: {enhanced_average:.3f}")

if enhanced_average > original_average:
    print("The enhanced summary achieved a higher average score.")
elif enhanced_average < original_average:
    print("The enhanced summary achieved a lower average score.")
else:
    print("Both summaries achieved the same average score.")

# 6 Display the enhanced summary and evaluation

print("ENHANCED SUMMARY")
print("=" * 60)
print(enhanced_result.Summary)

print("\nENHANCED EVALUATION")
print("=" * 60)
print(enhanced_evaluation.model_dump_json(indent=4))

"""
#7 Discussion: Did we get a better output?
The enhanced summary should be considered better if it demonstrates improvements in the evaluation results without introducing factual errors or violating the 1,000-token limit.

For example:
- If summarization improves, the revised summary may capture more of the report's important findings.
- If coherence improves, the information may be presented more clearly and logically.
- If tonality improves, the summary may follow Formal Academic Writing more consistently.
- If safety improves, the revised summary may contain fewer problematic or misleading statements.
However, an increase in the average score alone does not prove that the revised summary is factually superior. The original report should always be consulted when checking important claims.

Are these controls enough?
No. Although the self-correction process improves quality control, it is not sufficient to guarantee a completely reliable summary.
"""

Please, do not forget to add your comments.


# Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

## Submission Parameters

- The Submission Due Date is indicated in the [readme](../README.md#schedule) file.
- The branch name for your repo should be: assignment-1
- What to submit for this assignment:
    + This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
- What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    + Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

## Checklist

+ Created a branch with the correct naming convention.
+ Ensured that the repository is public.
+ Reviewed the PR description guidelines and adhered to them.
+ Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.
